# In vitro assay analysis


## Directory structure

Data is organized in the following structure:

```
data/
├── metadata.csv
├── images/
│   ├── 1_bf.tif
│   ├── 1_g.tif
│   ├── 1_r.tif
│   ├── 2_0.tif
│   └── ...

## metadata.csv example
figure_name,figure_id,channel,seeding_density,virus,cargo,dose_vg/well,image_time_h,receptor,include,notes
1_bf.tif,1,brightfield,0.5e5,AAV9,EGFP,0.5e10,24,EGFRvIII,1,
1_r.tif,1,brightfield,0.5e5,AAV9,EGFP,0.5e10,24,EGFRvIII,1,
```

In [18]:
import pandas as pd
from cellseg.metadata import metadata

data = '/Users/longwei/Desktop/janglab/figures/2026-05-30_in_vitro_assay/'
meta = data + 'metadata.csv'
result = data + 'in_vitro_assay_results.csv'
# Load or generate metadata
df_metadata = metadata(data_dir=data, verbose=True)
print("\nMetadata:")
df_metadata

Looking for metadata file at: /Users/longwei/Desktop/janglab/figures/2026-05-30_in_vitro_assay/metadata.csv
Looking for images in: /Users/longwei/Desktop/janglab/figures/2026-05-30_in_vitro_assay/
✓ Found and loaded metadata file (384 rows)

Metadata:


,figure_name,figure_id,experiment_id,magnification,channel,seeding_density,virus,cargo,dose_vg/well,image_time_h,receptor,include,notes
0,AAV6_Image001.tif,1,1,4x,bf,50000,AAV6,EGFP,2.500000e+10,24,EGFRvIII,1,NaN
1,AAV6_Image002.tif,2,1,4x,r,50000,AAV6,EGFP,2.500000e+10,24,EGFRvIII,1,NaN
2,AAV6_Image003.tif,3,1,4x,g,50000,AAV6,EGFP,2.500000e+10,24,EGFRvIII,1,NaN
3,AAV6_Image004.tif,4,2,4x,bf,50000,AAV6,EGFP,2.500000e+10,24,EGFRvIII,1,NaN
4,AAV6_Image005.tif,5,2,4x,r,50000,AAV6,EGFP,2.500000e+10,24,EGFRvIII,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
379,PHPeB_Image092.tif,380,127,10x,r,50000,PHPeB,EGFP,5.000000e+08,24,Ctrl,1,NaN
380,PHPeB_Image093.tif,381,127,10x,g,50000,PHPeB,EGFP,5.000000e+08,24,Ctrl,1,NaN
381,PHPeB_Image094.tif,382,128,10x,bf,50000,PHPeB,EGFP,5.000000e+08,24,Ctrl,1,NaN
382,PHPeB_Image095.tif,383,128,10x,r,50000,PHPeB,EGFP,5.000000e+08,24,Ctrl,1,NaN


In [19]:
from cellseg.quant import assay_rg_overlap_analysis

df_results = assay_rg_overlap_analysis(
    df_metadata,
    image_directory=data,
    output_file='rg_overlap_results.csv',
    bf_label='bf',
    r_label='r',
    g_label='g',
    r_channel=0,
    g_channel=1,
    min_r_intensity=0.06,
)

Processing RG overlap: 100%|██████████| 128/128 [04:15<00:00,  1.99s/it]


In [20]:
df_results
df_results.to_csv("/Users/longwei/Desktop/janglab/figures/2026-05-30_in_vitro_assay/df_results.csv", index=False)

In [21]:
import importlib
import cellseg.plot
importlib.reload(cellseg.plot)

from cellseg.plot import simple_bubble_plot
import bokeh.io
bokeh.io.output_notebook()

df_plot = df_results[df_results["status"] == "ok"].copy()
df_plot = df_plot[df_results["magnification"] == "4x"].copy()
df_plot["g_area_over_bf"] = df_plot["g_area"] / df_plot["bf_area"]

df_plot = (
    df_plot
    .groupby(["virus", "receptor", "dose_vg/well"])
    .mean(numeric_only=True)
    .reset_index()
)
print(df_plot)
for receptor in ["EGFRvIII", "Ctrl"]:
    p = simple_bubble_plot(
        df_plot[df_plot["receptor"] == receptor],
        x="virus",
        y="dose_vg/well",
        size="g_area_over_bf",
        color="g_intensity_all",
        x_order=["AAV6", "AAV9", "PHPeB"],
        y_order=[2.5e10, 5e9, 2.5e9, 5e8],
        size_range=(5, 38),
        width=400,
        height=360,
        title=receptor,
    )
    p.yaxis.axis_label = "Dose (v.g./well)"
    bokeh.io.show(p)


p = simple_bubble_plot(
    df_plot[df_plot["receptor"] == "EGFRvIII"],
    x="virus",
    y="dose_vg/well",
    size="overlap_g",
    color="g_intensity_overlap",
    x_order=["AAV6", "AAV9", "PHPeB"],
    y_order=[2.5e10, 5e9, 2.5e9, 5e8],
    size_range=(5, 38),
    width=400,
    height=360,
    title="EGFRvIII",
)
p.yaxis.axis_label = "Dose (v.g./well)"
bokeh.io.show(p)

Loading BokehJS ...

    virus  receptor  dose_vg/well  experiment_id  seeding_density  \
0    AAV6      Ctrl  5.000000e+08           23.0          50000.0   
1    AAV6      Ctrl  2.500000e+09           20.0          50000.0   
2    AAV6      Ctrl  5.000000e+09           17.0          50000.0   
3    AAV6      Ctrl  2.500000e+10           14.0          50000.0   
4    AAV6  EGFRvIII  5.000000e+08           11.0          50000.0   
5    AAV6  EGFRvIII  2.500000e+09            8.0          50000.0   
6    AAV6  EGFRvIII  5.000000e+09            5.0          50000.0   
7    AAV6  EGFRvIII  2.500000e+10            2.0          50000.0   
8    AAV9      Ctrl  5.000000e+08           71.0          50000.0   
9    AAV9      Ctrl  2.500000e+09           68.0          50000.0   
10   AAV9      Ctrl  5.000000e+09           65.0          50000.0   
11   AAV9      Ctrl  2.500000e+10           62.0          50000.0   
12   AAV9  EGFRvIII  5.000000e+08           59.0          50000.0   
13   AAV9  EGFRvIII  2.500000e+09 